In [9]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import hashlib
import html
import json
import re
import time

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Відкрий ноутбук усередині Bachelors-Thesis, де є data/raw.")

INPUT_FILES = [ROOT / "data" / "raw" / source / "articles.jsonl"
               for source in ["voxcheck", "stopfake", "bezbrehni", "gwara", "ukrinform"]]
MAX_ARTICLES_PER_FILE = None
CHECK_LANGUAGE = True
CHUNK_WORDS = 180
OVERLAP_WORDS = 40
TOP_K = 4
print("Корінь проєкту:", ROOT)


Корінь проєкту: d:\Bachelors-Thesis


In [10]:
def clean(value):
    if not isinstance(value, str):
        return ""
    value = re.sub(r"<[^>]+>", " ", value)
    return re.sub(r"\s+", " ", html.unescape(value)).strip()

def load_documents(paths):
    detector = None
    if CHECK_LANGUAGE:
        from lingua import Language, LanguageDetectorBuilder
        detector = LanguageDetectorBuilder.from_languages(
            Language.UKRAINIAN, Language.RUSSIAN, Language.ENGLISH,
            Language.BELARUSIAN, Language.POLISH).build()
    documents, seen, stats = [], set(), Counter()
    for path in paths:
        if not path.is_file():
            print("Пропущено відсутній файл:", path)
            continue
        accepted = 0
        with path.open(encoding="utf-8-sig") as stream:
            for line_number, line in enumerate(stream, 1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Некоректний JSON: {path}:{line_number}") from exc
                if not isinstance(record, dict):
                    continue
                if record.get("record_type", "factcheck_article") not in {
                    "factcheck_article", "news_article"}:
                    continue
                text = clean(record.get("article_text"))
                url = record.get("url")
                if len(text) < 250 or not isinstance(url, str) or not url.startswith(("https://", "http://")):
                    continue
                if detector is not None:
                    scores = detector.compute_language_confidence_values(text[:2000])
                    if not scores or scores[0].language != Language.UKRAINIAN or scores[0].value < 0.8:
                        stats["language_filtered"] += 1
                        continue
                fingerprint = hashlib.sha256(text.casefold().encode()).hexdigest()
                if fingerprint in seen:
                    stats["duplicates"] += 1
                    continue
                seen.add(fingerprint)
                documents.append({"doc_id": fingerprint[:24], "text": text,
                    "title": clean(record.get("title_original") or record.get("title")),
                    "url": url, "source": record.get("source") or path.parent.name,
                    "published_at": record.get("published_at")})
                stats[path.parent.name] += 1
                accepted += 1
                if MAX_ARTICLES_PER_FILE is not None and accepted >= MAX_ARTICLES_PER_FILE:
                    break
    if not documents:
        raise ValueError("Немає придатних статей. Перевір INPUT_FILES і поле article_text.")
    return documents, stats

documents, loading_stats = load_documents(INPUT_FILES)
print("Документів:", len(documents))
print(dict(loading_stats))


Документів: 32832
{'voxcheck': 9140, 'language_filtered': 40, 'stopfake': 5109, 'bezbrehni': 3316, 'gwara': 3275, 'ukrinform': 11992}


In [11]:
def make_chunks(documents):
    result = []
    step = CHUNK_WORDS - OVERLAP_WORDS
    if step <= 0:
        raise ValueError("Перекриття має бути меншим за розмір фрагмента.")
    for doc in documents:
        words = doc["text"].split()
        for start in range(0, len(words), step):
            part = " ".join(words[start:start + CHUNK_WORDS])
            if len(part) >= 100:
                result.append({k: v for k, v in doc.items() if k != "text"} |
                              {"chunk_id": f'{doc["doc_id"]}:{start}', "text": part})
            if start + CHUNK_WORDS >= len(words):
                break
    return result

started = time.perf_counter()
chunks = make_chunks(documents)
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2),
                            max_features=100000, sublinear_tf=True,
                            dtype=np.float32)
matrix = vectorizer.fit_transform([c["title"] + " " + c["text"] for c in chunks])
print("Фрагментів:", len(chunks))
print("Побудова індексу, с:", round(time.perf_counter() - started, 2))


Фрагментів: 100776
Побудова індексу, с: 93.43


In [12]:
def search_evidence(claim, top_k=TOP_K):
    claim = clean(claim)
    if len(claim) < 15:
        raise ValueError("Введи змістовне фактичне твердження, а не одне слово.")
    query = vectorizer.transform([claim])
    scores = (matrix @ query.T).toarray().ravel()
    order = np.argsort(-scores, kind="stable")
    results, per_doc = [], Counter()
    for index in order:
        if scores[index] <= 0:
            break
        chunk = chunks[int(index)]
        if per_doc[chunk["doc_id"]] >= 2:
            continue
        results.append(chunk | {"source_id": len(results) + 1,
                                "retrieval_score": round(float(scores[index]), 4)})
        per_doc[chunk["doc_id"]] += 1
        if len(results) >= top_k:
            break
    return results

def show_sources(results):
    if not results:
        print("Збігів не знайдено. Це не означає, що твердження неправдиве.")
    for item in results:
        print(f'\n[{item["source_id"]}] {item["title"]}')
        print("Джерело:", item["source"], "| Дата:", item["published_at"])
        print("URL:", item["url"])
        print("Лексична схожість:", item["retrieval_score"])
        print(item["text"])


In [19]:
CLAIM = "У Львові проїзд у трамваях стане безкоштовним для всіх студентів."

search_started = time.perf_counter()
retrieved = search_evidence(CLAIM)
search_seconds = time.perf_counter() - search_started
show_sources(retrieved)
print("\nЧас пошуку, с:", round(search_seconds, 3))



[1] Фейк: Жителі Сумської області змушені платити за «примусову евакуацію»
Джерело: stopfake | Дата: 2023-07-20T13:24:04+03:00
URL: https://www.stopfake.org/uk/fejk-zhiteli-sumskoyi-oblasti-zmusheni-platiti-za-primusovu-evakuatsiyu/
Лексична схожість: 0.1389
він курсуватиме за маршрутом Суми – Київ – Вінниця – Хмельницький – Івано-Франківськ. Проїзд у ньому буде безкоштовним . У Міністерстві з питань реінтеграції тимчасово окупованих територій зазначають , що на вокзалі у відділенні Укрпошти кожен громадянин після прибуття матиме право отримати допомогу в розмірі 3 тисячі гривень для дітей і осіб з інвалідністю та 2 тисячі гривень для всіх інших. Видання УНІАН повідомляє , повідомляє, що для внутрішньо переміщених осіб із прикордонних населених пунктів готують житло у безпечніших регіонах . Так, прийняти евакуйованих згодні у Чапліївці Шосткинського району – там для заселення ВПО переобладнують амбулаторію. Окрім цього, модульні містечка планують збудувати у Недригайлівській, Лохвицьк